In [116]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as st
import seaborn as sns
import math as mt
from skimage.io import imread, imsave

np.random.seed(123)

In [117]:
n = 100
beta = 0.95
theta_true = 22
data = st.uniform(loc=theta_true, scale=theta_true).rvs(size=n)
data

array([37.32232208, 28.29506537, 26.99073198, 34.12892492, 37.82831734,
       31.30834212, 43.57681236, 37.06625425, 32.58050183, 30.6265854 ,
       29.54991636, 38.03909356, 31.64858938, 23.31291373, 30.75697362,
       38.23589893, 26.01481807, 25.85993864, 33.69413022, 33.70020692,
       35.95682109, 40.68749947, 37.93801715, 35.44251723, 37.89375442,
       29.1050961 , 29.95935042, 27.02179108, 28.46170902, 35.88147472,
       24.02630868, 31.5414258 , 31.47898079, 32.86107215, 31.36826639,
       28.86974691, 31.37972875, 41.65456159, 42.7715204 , 33.04040687,
       35.72696494, 24.54360469, 28.9802806 , 31.12617666, 41.05880147,
       27.51001804, 32.62675381, 43.68231528, 33.42867262, 35.48367957,
       24.65383065, 40.17949761, 35.26732283, 33.99149614, 29.54080434,
       28.69065736, 31.17448864, 36.98861685, 41.26005052, 33.22929142,
       36.72490323, 34.89060416, 35.74787705, 36.84315912, 40.53153363,
       23.83028974, 38.80102251, 27.36066024, 26.27290513, 34.59

In [118]:
sample_mean = np.mean(data)
sample_max = np.max(data)

theta_mom = 2/3 * sample_mean  # метод моментов
theta_mle = (n+1) * sample_max / (2*n+1)  # метод максимального правдоподобия (смещенный)
sample_var = np.sum((data - sample_mean)**2) / n # выборочная дисперсия

In [119]:
gamma1 = ((1 + beta)/2)**(1/n)
gamma2 = ((1 - beta)/2)**(1/n)

accurate_left = sample_max / (1 + gamma1)
accurate_right = sample_max / (1 + gamma2)
accurate_length = accurate_right - accurate_left

print(f"Точный доверительный интервал для theta: {accurate_left:.6f} < theta < {accurate_right:.6f}")
print(f"Длина интервала: {accurate_length:.6f}")

Точный доверительный интервал для theta: 21.951722 < theta < 22.353732
Длина интервала: 0.402011


In [120]:
# Асимптотический доверительный интервал (ОММ)
z_crit = st.norm.ppf((1 + beta)/2)
std_error = 2/3 * np.sqrt(sample_var / n)

asym_left = theta_mom - z_crit * std_error
asym_right = theta_mom + z_crit * std_error
asymptotic_length = asym_right - asym_left

print("Асимптотический доверительный интервал для theta:", asym_left, "< theta <", asym_right)
print("Длина интервала:", asymptotic_length)

Асимптотический доверительный интервал для theta: 21.315581636146554 < theta < 22.726332178696893
Длина интервала: 1.410750542550339


In [121]:
# Непараметрический бутстраповский доверительный интервал для ОММ
N_bootstrap = 1000

def bootstrap_omm(data_sample, B):
    bootstrap_stats = []
    n_sample = len(data_sample)
    for _ in range(B):
        bootstrap_sample = np.random.choice(data_sample, size=n_sample, replace=True)        
        bootstrap_stats.append(2/3 * np.mean(bootstrap_sample) - theta_mom)
    return sorted(np.array(bootstrap_stats))

bootstrap_arr = bootstrap_omm(data, N_bootstrap)
quantile_idx_left = int((1 - beta)/2 * N_bootstrap - 1)
quantile_idx_right = int((1 + beta)/2 * N_bootstrap - 1)

bs_omm_left = theta_mom - bootstrap_arr[quantile_idx_right]
bs_omm_right = theta_mom - bootstrap_arr[quantile_idx_left]
bs_omm_length = bs_omm_right - bs_omm_left

print("Непараметрический бутстраповский доверительный интервал для theta(ОММ):", bs_omm_left, "< theta <", bs_omm_right)
print("l =", bs_omm_length)

Непараметрический бутстраповский доверительный интервал для theta(ОММ): 21.306620487672177 < theta < 22.767069878873414
l = 1.4604493912012373


In [122]:
# Непараметрический бутстраповский доверительный интервал для ОМП
def bootstrap_omp(data_sample, B):
    bootstrap_stats = []
    n_sample = len(data_sample)
    for _ in range(B):
        bootstrap_sample = np.random.choice(data_sample, size=n_sample, replace=True)        
        bootstrap_stats.append((n_sample + 1) * np.max(bootstrap_sample) / (2 * n_sample + 1) - theta_mle)
    return sorted(np.array(bootstrap_stats))

bootstrap_arr = bootstrap_omp(data, N_bootstrap)
quantile_idx_left = int((1 - beta)/2 * N_bootstrap)
quantile_idx_right = int((1 + beta)/2 * N_bootstrap)

bs_omp_left = theta_mle - bootstrap_arr[quantile_idx_right]
bs_omp_right = theta_mle - bootstrap_arr[quantile_idx_left]
bs_omp_length = bs_omp_right - bs_omp_left

print("Непараметрический бутстраповский доверительный интервал для theta(ОМП):", bs_omp_left, "< theta <", bs_omp_right)
print("l =", bs_omp_length)

Непараметрический бутстраповский доверительный интервал для theta(ОМП): 22.058142025271575 < theta < 22.624127033383846
l = 0.5659850081122713


In [123]:
# Сравнение доверительных интервалов
intervals_lengths = [
    (accurate_length, "Точный"),
    (asymptotic_length, "Асимптотический"),
    (bs_omm_length, "Бутстрап ОММ"),
    (bs_omp_length, "Бутстрап ОМП")
]
intervals_lengths.sort()

print("\nРейтинг доверительных интервалов:")
for i, (length, name) in enumerate(intervals_lengths, 1):
    print(f"{i}) {name} (l = {np.round(length, 3)})")


Рейтинг доверительных интервалов:
1) Точный (l = 0.402)
2) Бутстрап ОМП (l = 0.566)
3) Асимптотический (l = 1.411)
4) Бутстрап ОММ (l = 1.46)
